Projeto: Merca Data Platform

Squad 2 | Funções Utilitárias Centralizadas
> Este notebook centraliza todas as funções reutilizáveis do projeto. Deve ser chamado via %run pelos demais notebooks.

In [0]:
# Databricks notebook source
import os
import io
import logging
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

# ─────────────────────────────────────────────
# CONFIGURAÇÃO DE LOGS
# ─────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("squad2")

load_dotenv()

# Credenciais extraídas do arquivo .env
ADLS_CLIENT_ID       = os.getenv("ADLS_CLIENT_ID")
ADLS_TENANT_ID       = os.getenv("ADLS_TENANT_ID")
ADLS_CLIENT_SECRET   = os.getenv("ADLS_CLIENT_SECRET")
ADLS_STORAGE_ACCOUNT = os.getenv("ADLS_STORAGE_ACCOUNT")
ADLS_CONTAINER       = os.getenv("ADLS_CONTAINER")

PATHS = {
    "raw": "real-time-data",
    "bronze": "dbfs:/squad2/bronze",        # Salva no DBFS local por causa da limitação da conta Free
    "checkpoint": "/dbfs/squad2/checkpoints" # Checkpoint local
}

TABELAS_SQUAD2 = ["ecommerce_categorias", "ecommerce_itens_pedido", "ecommerce_produtos"]

# ─────────────────────────────────────────────
# FUNÇÕES DE CONEXÃO E LEITURA (AZURE SDK)
# ─────────────────────────────────────────────
def get_container_client():
    """Autentica no ADLS Gen2 via Service Principal usando o SDK da Azure."""
    credential = ClientSecretCredential(
        tenant_id=ADLS_TENANT_ID,
        client_id=ADLS_CLIENT_ID,
        client_secret=ADLS_CLIENT_SECRET
    )
    client = DataLakeServiceClient(
        account_url=f"https://{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net",
        credential=credential
    )
    return client.get_file_system_client(ADLS_CONTAINER)

def listar_snapshots() -> set:
    """Lista as pastas de snapshot no ADLS utilizando o Azure SDK."""
    base_path = PATHS["raw"]
    snapshots = set()
    try:
        container_client = get_container_client()
        paths = container_client.get_paths(path=base_path, recursive=True)
        for item in paths:
            partes = item.name.replace(base_path + "/", "").split("/")
            if len(partes) == 4 and item.is_directory:
                snapshots.add("/".join(partes))
    except Exception as e:
        log.error(f"Erro ao listar snapshots: {str(e)}")
    return snapshots

def ler_parquet_adls(snapshot_id: str, tabela: str):
    """Baixa o arquivo parquet do ADLS para a memória e converte em Spark DataFrame local."""
    base_path = PATHS["raw"]
    file_path = f"{base_path}/{snapshot_id}/{tabela}.parquet"
    
    container_client = get_container_client()
    file_client = container_client.get_file_client(file_path)
    
    bytes_data = file_client.download_file().readall()
    pdf = pd.read_parquet(io.BytesIO(bytes_data))
    
    return spark.createDataFrame(pdf)

# ─────────────────────────────────────────────
# GESTÃO DE CHECKPOINTS LOCAIS
# ─────────────────────────────────────────────
def ler_checkpoint(tabela: str) -> set:
    """Lê quais snapshots já foram processados para evitar duplicidade."""
    caminho = f"{PATHS['checkpoint']}/{tabela}_processed.txt"
    if os.path.exists(caminho):
        with open(caminho, "r") as f:
            return set(f.read().splitlines())
    return set()

def salvar_checkpoint(tabela: str, processados: set) -> None:
    """Salva a lista atualizada de snapshots processados."""
    caminho = f"{PATHS['checkpoint']}/{tabela}_processed.txt"
    os.makedirs(os.path.dirname(caminho), exist_ok=True)
    with open(caminho, "w") as f:
        f.write("\n".join(sorted(list(processados))))

# ─────────────────────────────────────────────
# FUNÇÕES DE AUDITORIA DE EXECUÇÃO
# ─────────────────────────────────────────────
def log_inicio(notebook: str) -> datetime:
    inicio = datetime.now()
    print(f"{'='*50}\nINÍCIO: {notebook}\nData  : {inicio.strftime('%Y-%m-%d %H:%M:%S')}\n{'='*50}")
    return inicio

def log_fim(notebook: str, inicio: datetime) -> None:
    duracao = (datetime.now() - inicio).seconds
    print(f"{'='*50}\nFIM: {notebook} | Tempo: {duracao}s\n{'='*50}")

# Validação automática das credenciais ao carregar o arquivo
def _validar_credenciais() -> None:
    if not all([ADLS_CLIENT_ID, ADLS_TENANT_ID, ADLS_CLIENT_SECRET, ADLS_STORAGE_ACCOUNT, ADLS_CONTAINER]):
        raise EnvironmentError("Credenciais ausentes no arquivo .env. Verifique as configurações.")
    print("Helpers carregados com sucesso! Modo Databricks Free ativo.")

_validar_credenciais()